# 01 — Experimentos com dados simulados

Situação 1 (normais multivariadas) e Situação 2 (distribuições heterogêneas). Inclui geração, TID-KMeans, DenStream, busca em grade, comparação online/offline, sensibilidade e 30 replicações.

**Execução:** abra este notebook e execute as células em ordem. É independente dos dados reais. A instalação do River está na primeira célula de código. As saídas anteriores foram removidas, sem modificar os cálculos.

## 0. Bibliotecas e funções de preparação

In [ ]:
%pip install river

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from river import cluster as river_cluster

SEED_SIMULACAO = 42
COLUNAS_SIMULADAS = [f"Var{i}" for i in range(1, 11)]

def obter_variaveis(df):
    """Usar exclusivamente variáveis preditoras; excluir rótulo e tempo."""
    return df.drop(columns=["Grupo", "tempo"], errors="ignore")

def padronizar_situacao(base, lotes):
    """Ajusta no período inicial e transforma os lotes, sem vazamento futuro."""
    colunas = list(obter_variaveis(base).columns)
    assert colunas == COLUNAS_SIMULADAS, f"Colunas inesperadas: {colunas}"
    scaler = StandardScaler().fit(base[colunas].to_numpy(float))
    def aplicar(df):
        numericas = pd.DataFrame(
            scaler.transform(df[colunas].to_numpy(float)),
            index=df.index, columns=colunas
        )
        outras = df.drop(columns=colunas).copy()
        return pd.concat([numericas, outras], axis=1).loc[:, df.columns]
    return aplicar(base), [aplicar(lote) for lote in lotes], scaler


## 1. Geração das duas situações simuladas

Na nomenclatura do TCC, **Situação 1 = normais multivariadas** e **Situação 2 = variáveis de distribuições distintas**. Cada situação possui 300 observações iniciais e dez lotes posteriores.

In [ ]:
def gerar_grupo(params, n):
    df = pd.DataFrame()

    for var, (dist, pars) in params.items():

        if dist == "normal":
            df[var] = np.random.normal(pars["loc"], pars["scale"], n)

        elif dist == "exponencial":
            df[var] = np.random.exponential(pars["scale"], n)

        elif dist == "poisson":
            df[var] = np.random.poisson(pars["lam"], n)

        elif dist == "bernoulli":
            df[var] = np.random.binomial(1, pars["p"], n)

        elif dist == "gamma":
            df[var] = np.random.gamma(pars["shape"], pars["scale"], n)

        elif dist == "uniforme":
            df[var] = np.random.uniform(pars["low"], pars["high"], n)

    return df

parametros = {
    0: {  # Grupo 0
        "Var1": ("normal", {"loc": 0, "scale": 1}),
        "Var2": ("exponencial", {"scale": 7}),
        "Var3": ("poisson", {"lam": 2}),
        "Var4": ("bernoulli", {"p": 0.2}),
        "Var5": ("gamma", {"shape": 2, "scale": 1}),
        "Var6": ("uniforme", {"low": 0, "high": 1}),
        "Var7": ("normal", {"loc": 1, "scale": 1}),
        "Var8": ("exponencial", {"scale": 3}),
        "Var9": ("poisson", {"lam": 3}),
        "Var10": ("uniforme", {"low": 1, "high": 2}),
    },

    1: {  # Grupo 1
        "Var1": ("normal", {"loc": 2, "scale": 1}),
        "Var2": ("exponencial", {"scale": 4}),
        "Var3": ("poisson", {"lam": 5}),
        "Var4": ("bernoulli", {"p": 0.5}),
        "Var5": ("gamma", {"shape": 5, "scale": 1}),
        "Var6": ("uniforme", {"low": 3, "high": 4}),
        "Var7": ("normal", {"loc": 3, "scale": 1}),
        "Var8": ("exponencial", {"scale": 2}),
        "Var9": ("poisson", {"lam": 6}),
        "Var10": ("uniforme", {"low": 4, "high": 5}),
    },

    2: {  # Grupo 2
        "Var1": ("normal", {"loc": 5, "scale": 1}),
        "Var2": ("exponencial", {"scale": 2}),
        "Var3": ("poisson", {"lam": 9}),
        "Var4": ("bernoulli", {"p": 0.8}),
        "Var5": ("gamma", {"shape": 9, "scale": 1}),
        "Var6": ("uniforme", {"low": 6, "high": 7}),
        "Var7": ("normal", {"loc": 6, "scale": 1}),
        "Var8": ("exponencial", {"scale": 1}),
        "Var9": ("poisson", {"lam": 10}),
        "Var10": ("uniforme", {"low": 7, "high": 8}),
    }
}

np.random.seed(42)

obs_por_grupo = 100
dados = []

for g in [0, 1, 2]:
    df_g = gerar_grupo(parametros[g], obs_por_grupo)
    df_g["Grupo"] = g
    dados.append(df_g)

df_final = pd.concat(dados, ignore_index=True)

# Tempo de chegada da base inicial
df_final["tempo"] = 0

print(df_final.shape)  # (300, 12)
print(df_final.head())

print(df_final.shape)  # (300, 11)
print(df_final.head())


In [ ]:
np.random.seed(42)

n_grupos = 3
obs_por_grupo = 100
n_variaveis = 10

dados = []
grupos = []

# Médias
medias = [
    np.zeros(n_variaveis),
    np.ones(n_variaveis) * 5,
    np.ones(n_variaveis) * 10
]

# Covariâncias

# Grupos 0 e 1 (sem correlação)
cov_identidade = np.identity(n_variaveis)

# Grupo 2 (com correlação)
cov_correlacionada = np.full((n_variaveis, n_variaveis), 0.5)
np.fill_diagonal(cov_correlacionada, 1)

# Gerar dados
for g in range(n_grupos):

    if g == 2:
        cov = cov_correlacionada
    else:
        cov = cov_identidade

    cluster = np.random.multivariate_normal(
        mean=medias[g],
        cov=cov,
        size=obs_por_grupo
    )

    dados.append(cluster)
    grupos.extend([g] * obs_por_grupo)

# Juntar
dados = np.vstack(dados)

df_normais = pd.DataFrame(
    dados,
    columns=[f"Var{i+1}" for i in range(n_variaveis)]
)

df_normais["Grupo"] = grupos

# Adicionar coluna de tempo
df_normais["tempo"] = 0

print(df_normais.head())
print(df_normais.shape)


In [ ]:
# Dias de chegada de cada lote
tempos_lotes = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

parametros_stream = parametros.copy()

parametros_stream[3] = {
    "Var1": ("normal", {"loc": 8, "scale": 1}),
    "Var2": ("exponencial", {"scale": 1}),
    "Var3": ("poisson", {"lam": 12}),
    "Var4": ("bernoulli", {"p": 0.9}),
    "Var5": ("gamma", {"shape": 12, "scale": 1}),
    "Var6": ("uniforme", {"low": 8, "high": 9}),
    "Var7": ("normal", {"loc": 9, "scale": 1}),
    "Var8": ("exponencial", {"scale": 0.5}),
    "Var9": ("poisson", {"lam": 13}),
    "Var10": ("uniforme", {"low": 9, "high": 10}),
}

batches_stream = []

for t in range(10):

    dados_batch = []

    if t < 3:
        grupos_ativos = [0, 1, 2]
        n_por_grupo = [7, 7, 6]
    else:
        grupos_ativos = [0, 1, 2, 3]
        n_por_grupo = [5, 5, 5, 18]

    for g, n in zip(grupos_ativos, n_por_grupo):

        df_g = gerar_grupo(parametros_stream[g], n)

        df_g["Grupo"] = g

        dados_batch.append(df_g)

    batch = pd.concat(dados_batch, ignore_index=True)

    # Acrescenta o dia de chegada do lote
    batch["tempo"] = tempos_lotes[t]

    batches_stream.append(batch)


In [ ]:
np.random.seed(42)

n_variaveis = 10
batches2 = []

# Tempos de chegada dos 10 lotes
tempos = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

for t in range(10):

    if t < 3:
        # antes do novo cluster
        dados = np.vstack([
            np.random.normal(0, 1, (15, n_variaveis)),
            np.random.normal(5, 1, (25, n_variaveis)),
            np.random.normal(10, 1, (10, n_variaveis))
        ])

        grupos = [0]*15 + [1]*25 + [2]*10

    else:
        # aparece novo cluster
        dados = np.vstack([
            np.random.normal(0, 1, (5, n_variaveis)),
            np.random.normal(5, 1, (10, n_variaveis)),
            np.random.normal(10, 1, (10, n_variaveis)),
            np.random.normal(15, 1, (25, n_variaveis))
        ])

        grupos = [0]*5 + [1]*10 + [2]*10 + [3]*25

    df = pd.DataFrame(
        dados,
        columns=[f"Var{i}" for i in range(1, n_variaveis+1)]
    )

    df["Grupo"] = grupos  # adiciona grupo

    # Adicionar coluna de tempo ao lote
    df["tempo"] = tempos[t]

    batches2.append(df)


In [ ]:
# Guardar os dados ORIGINAIS: as funções de avaliação padronizam internamente
# com as 300 observações iniciais, inclusive dentro de cada replicação.
SITUACOES_BRUTAS = {
    "Situação 1": (df_normais.copy(deep=True), [b.copy(deep=True) for b in batches2]),
    "Situação 2": (df_final.copy(deep=True), [b.copy(deep=True) for b in batches_stream]),
}
SITUACOES_PADRONIZADAS = {}
for nome, (base, lotes) in SITUACOES_BRUTAS.items():
    base_pad, lotes_pad, scaler = padronizar_situacao(base, lotes)
    SITUACOES_PADRONIZADAS[nome] = (base_pad, lotes_pad, scaler)
    print(f"{nome}: inicial={len(base)}, lotes={len(lotes)}, total={len(base) + sum(map(len, lotes))}")
    print("  médias X0:", np.round(base_pad[COLUNAS_SIMULADAS].mean().to_numpy(), 7))
    print("  desvios X0 (ddof=0):", np.round(base_pad[COLUNAS_SIMULADAS].std(ddof=0).to_numpy(), 7))
    assert np.allclose(base_pad[COLUNAS_SIMULADAS].mean(), 0, atol=1e-12)
    assert np.allclose(base_pad[COLUNAS_SIMULADAS].std(ddof=0), 1, atol=1e-12)
    assert base_pad['Grupo'].equals(base['Grupo'])
    assert base_pad['tempo'].equals(base['tempo'])
    for bruto, pad in zip(lotes, lotes_pad):
        assert bruto['Grupo'].equals(pad['Grupo'])
        assert bruto['tempo'].equals(pad['tempo'])
        assert np.allclose(scaler.transform(bruto[COLUNAS_SIMULADAS].to_numpy(float)), pad[COLUNAS_SIMULADAS])


## 2. Implementação do TID-KMeans e funções de avaliação


In [ ]:
import numpy as np
from sklearn.cluster import KMeans


class TIDKMeans:

    def __init__(
        self,
        n_clusters,
        lambda_decay,
        fator_limiar
    ):

        self.n_clusters = n_clusters
        self.lambda_decay = lambda_decay
        self.fator_limiar = fator_limiar

        # Tempo do último lote processado
        self.ultimo_tempo = 0

    # ==================================================
    # FASE INICIAL: K-MEANS
    # ==================================================

    def fit_initial(self, X0):

        kmeans = KMeans(
            n_clusters=self.n_clusters,
            random_state=123,
            n_init=50
        )

        labels = kmeans.fit_predict(X0)

        self.labels_initial = labels
        self.clusters = []


        # Informações iniciais de cada cluster
        for j in range(self.n_clusters):

            pontos_cluster = X0[labels == j]

            N_j = len(pontos_cluster)

            LS_j = pontos_cluster.sum(axis=0)

            c_j = LS_j / N_j

            # Distâncias ao centróide
            distancias = np.linalg.norm(
                pontos_cluster - c_j,
                axis=1
            )

            # Limiar específico do cluster
            # Utiliza o fator definido na inicialização
            D_j = self.fator_limiar * distancias.max()

            self.clusters.append({
                "LS": LS_j,
                "N": N_j,
                "centroide": c_j,
                "limiar": D_j
            })

        # Média dos limiares iniciais
        self.limiar_medio = np.mean([
            c["limiar"] for c in self.clusters
        ])

        # A base inicial corresponde ao tempo 0
        self.ultimo_tempo = 0

    # ==================================================
    # DECAIMENTO TEMPORAL DE TODOS OS CLUSTERS
    # ==================================================

    def aplicar_decaimento(self, tempo_atual):

        # Diferença em dias desde o último lote
        delta_t = tempo_atual - self.ultimo_tempo

        if delta_t < 0:
            raise ValueError(
                "O tempo atual não pode ser menor "
                "que o tempo do último lote processado."
            )

        # Fator de decaimento
        peso = 2.0 ** (
             -self.lambda_decay * delta_t
            )

        # Aplicar decaimento em TODOS os clusters
        for cluster in self.clusters:

            cluster["LS"] *= peso
            cluster["N"] *= peso

            # Atualizar centróide
            cluster["centroide"] = (
                cluster["LS"] / cluster["N"]
            )

        # Atualiza o tempo global do fluxo
        self.ultimo_tempo = tempo_atual

    # ==================================================
    # PROCESSAR UM LOTE
    # ==================================================

    def process_batch(self, X_batch, tempo):

        # 1. Aplicar decaimento em todos os clusters
        self.aplicar_decaimento(tempo)

        labels_pred = []

        # 2. Inserir as observações do lote
        for x_t in X_batch:

            label = self.process_one(x_t)

            labels_pred.append(label)

        return np.array(labels_pred)

    # ==================================================
    # PROCESSAR UMA OBSERVAÇÃO
    # ==================================================

    def process_one(self, x_t):

        # Calcular distância aos centróides atuais
        distancias = []

        for idx, cluster in enumerate(self.clusters):

            d = np.linalg.norm(
                x_t - cluster["centroide"]
            )

            distancias.append((idx, d))

        # Ordenar pela menor distância
        distancias.sort(key=lambda x: x[1])

        # Tentar inserir no cluster mais próximo
        for idx, d in distancias:

            cluster = self.clusters[idx]

            if d <= cluster["limiar"]:

                # Acrescentar a observação
                cluster["LS"] += x_t
                cluster["N"] += 1

                # Atualizar centróide
                cluster["centroide"] = (
                    cluster["LS"] / cluster["N"]
                )

                return idx

        # Nenhum cluster aceitou a observação:
        # criar um novo cluster
        novo_cluster = {
            "LS": x_t.copy(),
            "N": 1.0,
            "centroide": x_t.copy(),
            "limiar": self.limiar_medio
        }

        self.clusters.append(novo_cluster)

        return len(self.clusters) - 1

    # ==================================================
    # FUNÇÕES AUXILIARES
    # ==================================================

    def get_centroids(self):

        return np.array([
            c["centroide"] for c in self.clusters
        ])

    def get_number_clusters(self):

        return len(self.clusters)

    def get_cluster_info(self):

        return self.clusters


In [ ]:
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd

from sklearn.metrics import adjusted_rand_score
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from river import cluster as river_cluster

# ==========================================================
# FUNÇÕES AUXILIARES
# ==========================================================

def obter_variaveis(df):
    """Retorna somente as variáveis usadas na clusterização."""
    colunas_excluir = ["Grupo", "tempo"]
    return df.drop(columns=colunas_excluir, errors="ignore")


def obter_centros_denstream(modelo, colunas):
    """
    Obtém os centros dos p-micro-clusters do DenStream,
    na mesma ordem das colunas usadas como atributos.
    """
    centros = []

    for mc in modelo.p_micro_clusters.values():
        centro = mc.calc_center(modelo.timestamp)

        vetor = np.array(
            [centro.get(coluna, 0.0) for coluna in colunas],
            dtype=float
        )

        centros.append(vetor)

    if len(centros) == 0:
        return np.empty((0, len(colunas)))

    return np.vstack(centros)


def alinhar_rotulos_denstream(
    centros_atuais,
    centros_anteriores,
    ids_anteriores,
    proximo_id,
    distancia_maxima
):
    """
    Tenta manter IDs de clusters consistentes entre lotes,
    associando centros atuais aos centros anteriores por
    distância mínima.

    Centros sem correspondência recebem novos IDs.
    """
    n_atual = len(centros_atuais)
    ids_atuais = np.full(n_atual, -1, dtype=int)

    if n_atual == 0:
        return ids_atuais, proximo_id

    if len(centros_anteriores) > 0:
        distancias = cdist(centros_atuais, centros_anteriores)

        linhas, colunas = linear_sum_assignment(distancias)

        for i_atual, i_anterior in zip(linhas, colunas):
            if distancias[i_atual, i_anterior] <= distancia_maxima:
                ids_atuais[i_atual] = ids_anteriores[i_anterior]

    for i in range(n_atual):
        if ids_atuais[i] == -1:
            ids_atuais[i] = proximo_id
            proximo_id += 1

    return ids_atuais, proximo_id


def predizer_por_centros(X, centros):
    """Associa cada observação ao centro mais próximo."""
    if len(centros) == 0:
        raise ValueError(
            "O DenStream não possui p-micro-clusters "
            "disponíveis para realizar a predição."
        )

    distancias = cdist(X, centros)
    return distancias.argmin(axis=1)


# ==========================================================
# AVALIAÇÃO DO DYNAMIC K-MEANS
# ==========================================================

def avaliar_tid_sim(
    df_base, lotes, lambda_decay, fator_limiar, n_clusters=2
):
    # Padronização ajustada exclusivamente à base inicial desta replicação
    X0_df = obter_variaveis(df_base)
    scaler = StandardScaler().fit(X0_df.to_numpy(dtype=float))
    X0 = scaler.transform(X0_df.to_numpy(dtype=float))
    y0 = df_base["Grupo"].to_numpy()

    modelo = TIDKMeans(
        n_clusters,
        lambda_decay=lambda_decay,
        fator_limiar=fator_limiar
    )

    modelo.fit_initial(X0)

    # Avaliação da base inicial
    pred_inicial = modelo.labels_initial
    ari_inicial = adjusted_rand_score(y0, pred_inicial)

    # Armazenamento dos resultados
    resultados_lotes = []
    y_true_stream = []
    y_pred_stream = []

    y_true_global = list(y0)
    y_pred_global = list(pred_inicial)

    # Processamento dos lotes
    for i, lote in enumerate(lotes, start=1):
        X_lote = scaler.transform(obter_variaveis(lote).to_numpy(dtype=float))
        y_lote = lote["Grupo"].to_numpy()

        tempo_lote = (
            lote["tempo"].iloc[0]
            if "tempo" in lote.columns
            else i
        )

        pred = modelo.process_batch(X_lote, tempo_lote)

        ari_lote = adjusted_rand_score(y_lote, pred)

        resultados_lotes.append({
            "Lote": i,
            "Tempo": tempo_lote,
            "ARI": ari_lote,
            "Numero_clusters": modelo.get_number_clusters()
        })

        y_true_stream.extend(y_lote)
        y_pred_stream.extend(pred)

        y_true_global.extend(y_lote)
        y_pred_global.extend(pred)

    # ARI considerando somente as observações do fluxo
    ari_stream = adjusted_rand_score(
        y_true_stream, y_pred_stream
    )

    # ARI incluindo a base inicial e todo o fluxo
    ari_global = adjusted_rand_score(
        y_true_global, y_pred_global
    )

    return {
        "ARI_inicial": ari_inicial,
        "ARI_stream": ari_stream,
        "ARI_global_com_base": ari_global,
        "ARI_lotes": resultados_lotes,
        "Numero_clusters_final": modelo.get_number_clusters(),
        "y_inicial": y0, "pred_inicial": np.asarray(pred_inicial),
        "y_stream": np.asarray(y_true_stream),
        "pred_stream": np.asarray(y_pred_stream),
    }



# ==========================================================
# AVALIAÇÃO DO DENSTREAM
# ==========================================================

def avaliar_denstream_sim(df_base, lotes, config):

    # ------------------------------------------------------
    # 1. Inicializar e treinar com a base inicial
    # ------------------------------------------------------

    X0_df = obter_variaveis(df_base)
    colunas = list(X0_df.columns)

    # O mesmo protocolo aplicado ao TID-KMeans
    scaler = StandardScaler().fit(X0_df.to_numpy(dtype=float))
    X0 = scaler.transform(X0_df.to_numpy(dtype=float))
    y0 = df_base["Grupo"].to_numpy()

    modelo = river_cluster.DenStream(
        epsilon=config["epsilon"],
        beta=config["beta"],
        mu=config["mu"],
        decaying_factor=config["decaying_factor"],
        n_samples_init=config["n_samples_init"],
        stream_speed=config["stream_speed"]
    )

    # O River recebe cada observação como um dicionário
    X0_pad_df = pd.DataFrame(X0, columns=colunas)
    for registro in X0_pad_df.to_dict(orient="records"):
        modelo.learn_one(registro)

    # Centros disponíveis após o treinamento inicial
    centros = obter_centros_denstream(modelo, colunas)

    if len(centros) == 0:
        raise ValueError(
            "O DenStream não formou p-micro-clusters "
            "após o treinamento inicial."
        )

    # IDs iniciais dos clusters
    ids_centros = np.arange(len(centros), dtype=int)
    proximo_id = len(centros)

    # Avaliação retrospectiva da base inicial
    pred_inicial = predizer_por_centros(X0, centros)
    ari_inicial = adjusted_rand_score(y0, pred_inicial)

    y_true_global = list(y0)
    y_pred_global = list(ids_centros[pred_inicial])

    y_true_stream = []
    y_pred_stream = []
    resultados_lotes = []

    # ------------------------------------------------------
    # 2. Processar o fluxo lote a lote
    # ------------------------------------------------------

    for i, lote in enumerate(lotes, start=1):

        X_lote_df = obter_variaveis(lote)
        X_lote = scaler.transform(X_lote_df.to_numpy(dtype=float))
        y_lote = lote["Grupo"].to_numpy()

        tempo_lote = (
            lote["tempo"].iloc[0]
            if "tempo" in lote.columns
            else i
        )

        # Predizer usando os centros disponíveis ANTES
        # de aprender as observações deste lote
        pred_indices = predizer_por_centros(X_lote, centros)
        pred = ids_centros[pred_indices]

        ari_lote = adjusted_rand_score(y_lote, pred)

        y_true_stream.extend(y_lote)
        y_pred_stream.extend(pred)

        y_true_global.extend(y_lote)
        y_pred_global.extend(pred)

        # Aprender as observações do lote, uma a uma
        X_lote_pad_df = pd.DataFrame(X_lote, columns=colunas)
        for registro in X_lote_pad_df.to_dict(orient="records"):
            modelo.learn_one(registro)

        # Obter os centros atualizados para o próximo lote
        centros_novos = obter_centros_denstream(modelo, colunas)

        if len(centros_novos) > 0:

            ids_novos, proximo_id = alinhar_rotulos_denstream(
                centros_atuais=centros_novos,
                centros_anteriores=centros,
                ids_anteriores=ids_centros,
                proximo_id=proximo_id,
                distancia_maxima=config["epsilon"]
            )

            centros = centros_novos
            ids_centros = ids_novos

        # Quantidade de p-micro-clusters após aprender o lote
        resultados_lotes.append({
            "Lote": i,
            "Tempo": tempo_lote,
            "ARI": ari_lote,
            "Numero_clusters": len(modelo.p_micro_clusters)
        })

    # ------------------------------------------------------
    # 3. Calcular os resultados gerais
    # ------------------------------------------------------

    ari_stream = adjusted_rand_score(
        y_true_stream,
        y_pred_stream
    )

    ari_global = adjusted_rand_score(
        y_true_global,
        y_pred_global
    )

    return {
        "ARI_inicial": ari_inicial,
        "ARI_stream": ari_stream,
        "ARI_global_com_base": ari_global,
        "ARI_lotes": resultados_lotes,
        "Numero_clusters_final": len(modelo.p_micro_clusters),
        "y_inicial": y0, "pred_inicial": np.asarray(pred_inicial),
        "y_stream": np.asarray(y_true_stream),
        "pred_stream": np.asarray(y_pred_stream),
    }


## 3. Busca em grade — TID-KMeans e DenStream

A grade do decaimento foi preservada para ambos(`[0.01, 0.1, 0.5, 1]`). TID-KMeans: 204 combinações por situação; DenStream: 144 por situação. A seleção utiliza ARI global e os mesmos dados brutos são fornecidos a ambos os procedimentos, cada qual padronizando apenas pelo período inicial.

In [ ]:
LAMBDA_GRID = [0.01, 0.1, 0.5, 1]
ALPHA_GRID = np.round(np.arange(1.00, 1.501, 0.01), 2)
EPSILON_GRID = [2, 5, 8, 11]
MU_GRID = [2, 5, 8]
BETA_GRID = [0.1, 0.2, 0.5]
DECAY_GRID = [0.01, 0.1, 0.5, 1]

print('TID-KMeans:', len(LAMBDA_GRID)*len(ALPHA_GRID), 'combinações por situação')
print('DenStream:', len(EPSILON_GRID)*len(MU_GRID)*len(BETA_GRID)*len(DECAY_GRID), 'combinações por situação')

def grid_tid_situacoes(situacoes):
    linhas = []
    for nome, (base, lotes) in situacoes.items():
        for lam, alpha in itertools.product(LAMBDA_GRID, ALPHA_GRID):
            r = avaliar_tid_sim(base, lotes, float(lam), float(alpha), n_clusters=3)
            linhas.append({
                'Situação': nome, 'lambda_decay': float(lam), 'fator_limiar': float(alpha),
                'ARI_global': r['ARI_global_com_base'], 'ARI_stream': r['ARI_stream'],
                'Clusters_finais': r['Numero_clusters_final']
            })
    return pd.DataFrame(linhas).sort_values(['Situação','ARI_global'], ascending=[True, False]).reset_index(drop=True)

def grid_denstream_situacoes(situacoes):
    linhas = []
    for nome, (base, lotes) in situacoes.items():
        for eps, mu, beta, d in itertools.product(EPSILON_GRID, MU_GRID, BETA_GRID, DECAY_GRID):
            config = dict(epsilon=eps, mu=mu, beta=beta, decaying_factor=d, n_samples_init=300, stream_speed=1)
            try:
                r = avaliar_denstream_sim(base, lotes, config)
                linhas.append({
                    'Situação':nome, **config, 'ARI_global':r['ARI_global_com_base'],
                    'ARI_stream':r['ARI_stream'], 'Clusters_finais':r['Numero_clusters_final'], 'Erro':''
                })
            except (ValueError, OverflowError, ZeroDivisionError) as exc:
                linhas.append({
                    'Situação':nome, **config, 'ARI_global':np.nan,
                    'ARI_stream':np.nan, 'Clusters_finais':np.nan, 'Erro':str(exc)
                })
    return pd.DataFrame(linhas).sort_values(['Situação','ARI_global'], ascending=[True, False]).reset_index(drop=True)


In [ ]:
grid_tid = grid_tid_situacoes(SITUACOES_BRUTAS)
print(grid_tid.groupby('Situação').head(5).round(4).to_string(index=False))
grid_tid.to_csv('grid_tid_padronizado.csv',index=False)


In [ ]:
grid_denstream = grid_denstream_situacoes(SITUACOES_BRUTAS)
print(grid_denstream.groupby('Situação').head(5).round(4).to_string(index=False))
grid_denstream.to_csv('grid_denstream_padronizado.csv',index=False)


In [ ]:
for metodo, tabela in [
    ("TID-KMeans", grid_tid),
    ("DenStream", grid_denstream)
]:
    resumo = (
        tabela.groupby("Situação")["ARI_global"]
        .agg(["min", "max"])
    )
    resumo["amplitude"] = resumo["max"] - resumo["min"]

    print(f"\n{metodo}")
    print(resumo.round(4))


## 4. Execuções individuais com os parâmetros selecionados

As configurações são extraídas dos próprios grids. Assim, matrizes, ARIs e replicações usam a mesma escolha de hiperparâmetros.

In [ ]:
def melhores_configs(grid_tid, grid_denstream):
    escolhas = {}
    for nome in SITUACOES_BRUTAS:
        t = grid_tid[(grid_tid['Situação']==nome) & grid_tid['ARI_global'].notna()].iloc[0]
        d = grid_denstream[(grid_denstream['Situação']==nome) & grid_denstream['ARI_global'].notna()].iloc[0]
        escolhas[nome] = {
            'TID': dict(lambda_decay=float(t['lambda_decay']), fator_limiar=float(t['fator_limiar'])),
            'DenStream': dict(
                epsilon=float(d['epsilon']), mu=int(d['mu']), beta=float(d['beta']),
                decaying_factor=float(d['decaying_factor']), n_samples_init=300, stream_speed=1
            )
        }
    return escolhas

ESCOLHAS_SIM = melhores_configs(grid_tid, grid_denstream)
for nome, escolhas in ESCOLHAS_SIM.items():
    print(nome, escolhas)

RESULTADOS_INDIVIDUAIS = {}
for nome, (base, lotes) in SITUACOES_BRUTAS.items():
    config = ESCOLHAS_SIM[nome]
    resultado_tid = avaliar_tid_sim(base, lotes, **config['TID'], n_clusters=3)
    resultado_den = avaliar_denstream_sim(base, lotes, config['DenStream'])
    RESULTADOS_INDIVIDUAIS[nome] = {'TID-KMeans': resultado_tid, 'DenStream': resultado_den}
    for metodo, resultado in RESULTADOS_INDIVIDUAIS[nome].items():
        print(f"\n{nome} — {metodo}: inicial={resultado['ARI_inicial']:.4f}, fluxo={resultado['ARI_stream']:.4f}, global={resultado['ARI_global_com_base']:.4f}")
        print(pd.DataFrame(resultado['ARI_lotes']).round(4).to_string(index=False))
        print('Matriz inicial:')
        print(pd.crosstab(resultado['y_inicial'], resultado['pred_inicial']))
        ultimo = lotes[-1]
        n_ultimo = len(ultimo)
        print('Matriz do último lote:')
        print(pd.crosstab(resultado['y_stream'][-n_ultimo:], resultado['pred_stream'][-n_ultimo:]))


### ARI por lote — comparação individual nas situações simuladas

Os gráficos utilizam as execuções individuais com os parâmetros escolhidos pela busca em grade. São mostrados os ARIs de cada lote, sem incluir o ARI global ou o conjunto inicial. Os dados das duas situações foram padronizados usando apenas a base inicial.

In [ ]:
# Gráficos das execuções individuais (não representam médias das replicações)
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)

for ax, nome in zip(axes, ["Situação 1", "Situação 2"]):
    resultados = RESULTADOS_INDIVIDUAIS[nome]
    for metodo, resultado in resultados.items():
        tabela = pd.DataFrame(resultado["ARI_lotes"]).sort_values("Lote")
        ax.plot(tabela["Lote"], tabela["ARI"], marker="o",
                linewidth=2, markersize=5, label=metodo)
    ax.set_title(nome)
    ax.set_xlabel("Lote")
    ax.set_xticks(range(1, 11))
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.25)
    ax.legend()

axes[0].set_ylabel("Índice de Rand Ajustado (ARI)")
fig.suptitle("ARI por lote — TID-KMeans e DenStream", fontsize=13)
fig.tight_layout()
plt.show()

# Opcional: salvar a figura para inclusão no TCC
fig.savefig("ari_por_lote_situacoes_simuladas.png", dpi=300, bbox_inches="tight")


## 5. DenStream: verificação adicional da etapa offline

Esta comparação usa a mesma padronização e as configurações selecionadas.

In [ ]:
import numpy as np
import pandas as pd

from river import cluster
from scipy.spatial.distance import cdist
from sklearn.metrics import adjusted_rand_score


def comparar_denstream_online_offline(df_base, lotes, config):
    """
    Compara:
    - Online: centro do p-micro-cluster mais próximo.
    - Offline: macrocluster produzido pelo DenStream do River.

    A avaliação ocorre após a atualização de cada lote.
    """

    # As entradas desta função precisam estar padronizadas com o scaler da base inicial.
    modelo = cluster.DenStream(**config)

    colunas = df_base.drop(columns=["Grupo", "tempo"], errors="ignore").columns.tolist()
    resultados = []

    def avaliar_etapa(df, etapa):
        X = df[colunas].to_numpy(dtype=float)
        y = df["Grupo"].to_numpy()

        # ---------------------------
        # ONLINE: p-micro-clusters
        # ---------------------------
        centros_micro = []

        for mc in modelo.p_micro_clusters.values():
            centro = mc.calc_center(modelo.timestamp)
            centros_micro.append([
                centro.get(c, 0.0) for c in colunas
            ])

        if centros_micro:
            centros_micro = np.asarray(centros_micro)
            pred_online = cdist(
                X, centros_micro
            ).argmin(axis=1)
        else:
            pred_online = np.full(len(X), -1)

        # ---------------------------
        # OFFLINE: macroclusters
        # ---------------------------
        # predict_one executa a etapa offline
        # a partir do estado atual do modelo.
        if len(X):
            modelo.predict_one(
                dict(zip(colunas, X[0]))
            )

        centros_macro_dict = modelo.centers
        centros_macro = np.asarray([
            [centro.get(c, 0.0) for c in colunas]
            for centro in centros_macro_dict.values()
        ]).reshape(-1, len(colunas))

        if len(centros_macro):
            pred_offline = cdist(
                X, centros_macro
            ).argmin(axis=1)
        else:
            pred_offline = np.full(len(X), -1)

        ari_online = adjusted_rand_score(y, pred_online)
        ari_offline = adjusted_rand_score(y, pred_offline)

        resultados.append({
            "Etapa": etapa,
            "N_micro": len(centros_micro),
            "N_macro": len(centros_macro),
            "ARI_online": ari_online,
            "ARI_offline": ari_offline,
            "Diferenca": ari_offline - ari_online
        })

    # Inicialização
    for x in df_base[colunas].to_dict(orient="records"):
        modelo.learn_one(x)

    avaliar_etapa(df_base, "Inicial")

    # Fluxo: atualiza antes de avaliar cada lote
    for t, lote in enumerate(lotes, start=1):
        for x in lote[colunas].to_dict(orient="records"):
            modelo.learn_one(x)

        avaliar_etapa(lote, f"Lote {t}")

    return pd.DataFrame(resultados)


In [ ]:
RESULTADOS_OFFLINE_SIM = {}
for nome, (base_pad, lotes_pad, _) in SITUACOES_PADRONIZADAS.items():
    tab = comparar_denstream_online_offline(base_pad, lotes_pad, ESCOLHAS_SIM[nome]['DenStream'])
    RESULTADOS_OFFLINE_SIM[nome] = tab
    print(f"\n{nome} — comparação pós-atualização (online/offline):")
    print(tab.round(4).to_string(index=False))
    print('Maior diferença absoluta de ARI:', tab['Diferenca'].abs().max())


## 6. Replicações independentes (30 sementes)

Cada repetição gera uma nova base e reajusta o `StandardScaler` apenas na base inicial daquela repetição. A configuração de hiperparâmetros vem da busca em grade anterior (não é reajustada a cada repetição).

In [ ]:
import numpy as np
import pandas as pd


def gerar_grupo(params, n):
    df = pd.DataFrame()

    for var, (dist, pars) in params.items():

        if dist == "normal":
            df[var] = np.random.normal(
                pars["loc"], pars["scale"], n
            )

        elif dist == "exponencial":
            df[var] = np.random.exponential(
                pars["scale"], n
            )

        elif dist == "poisson":
            df[var] = np.random.poisson(
                pars["lam"], n
            )

        elif dist == "bernoulli":
            df[var] = np.random.binomial(
                1, pars["p"], n
            )

        elif dist == "gamma":
            df[var] = np.random.gamma(
                pars["shape"], pars["scale"], n
            )

        elif dist == "uniforme":
            df[var] = np.random.uniform(
                pars["low"], pars["high"], n
            )

    return df


def gerar_base_1(seed):

    # Reproduzibilidade da replicação
    np.random.seed(seed)

    parametros = {
        0: {
            "Var1": ("normal", {"loc": 0, "scale": 1}),
            "Var2": ("exponencial", {"scale": 7}),
            "Var3": ("poisson", {"lam": 2}),
            "Var4": ("bernoulli", {"p": 0.2}),
            "Var5": ("gamma", {"shape": 2, "scale": 1}),
            "Var6": ("uniforme", {"low": 0, "high": 1}),
            "Var7": ("normal", {"loc": 1, "scale": 1}),
            "Var8": ("exponencial", {"scale": 3}),
            "Var9": ("poisson", {"lam": 3}),
            "Var10": ("uniforme", {"low": 1, "high": 2}),
        },

        1: {
            "Var1": ("normal", {"loc": 2, "scale": 1}),
            "Var2": ("exponencial", {"scale": 4}),
            "Var3": ("poisson", {"lam": 5}),
            "Var4": ("bernoulli", {"p": 0.5}),
            "Var5": ("gamma", {"shape": 5, "scale": 1}),
            "Var6": ("uniforme", {"low": 3, "high": 4}),
            "Var7": ("normal", {"loc": 3, "scale": 1}),
            "Var8": ("exponencial", {"scale": 2}),
            "Var9": ("poisson", {"lam": 6}),
            "Var10": ("uniforme", {"low": 4, "high": 5}),
        },

        2: {
            "Var1": ("normal", {"loc": 5, "scale": 1}),
            "Var2": ("exponencial", {"scale": 2}),
            "Var3": ("poisson", {"lam": 9}),
            "Var4": ("bernoulli", {"p": 0.8}),
            "Var5": ("gamma", {"shape": 9, "scale": 1}),
            "Var6": ("uniforme", {"low": 6, "high": 7}),
            "Var7": ("normal", {"loc": 6, "scale": 1}),
            "Var8": ("exponencial", {"scale": 1}),
            "Var9": ("poisson", {"lam": 10}),
            "Var10": ("uniforme", {"low": 7, "high": 8}),
        }
    }

    # -------------------------------
    # Base inicial: 300 observações
    # -------------------------------
    dados_iniciais = []

    for g in [0, 1, 2]:
        df_g = gerar_grupo(parametros[g], 100)
        df_g["Grupo"] = g
        dados_iniciais.append(df_g)

    df_base = pd.concat(
        dados_iniciais, ignore_index=True
    )

    df_base["tempo"] = 0

    # -------------------------------
    # Parâmetros do novo grupo
    # -------------------------------
    parametros_stream = parametros.copy()

    parametros_stream[3] = {
        "Var1": ("normal", {"loc": 8, "scale": 1}),
        "Var2": ("exponencial", {"scale": 1}),
        "Var3": ("poisson", {"lam": 12}),
        "Var4": ("bernoulli", {"p": 0.9}),
        "Var5": ("gamma", {"shape": 12, "scale": 1}),
        "Var6": ("uniforme", {"low": 8, "high": 9}),
        "Var7": ("normal", {"loc": 9, "scale": 1}),
        "Var8": ("exponencial", {"scale": 0.5}),
        "Var9": ("poisson", {"lam": 13}),
        "Var10": ("uniforme", {"low": 9, "high": 10}),
    }

    # -------------------------------
    # Geração dos 10 lotes
    # -------------------------------
    tempos_lotes = list(range(1, 11))
    lotes = []

    for t in range(10):

        dados_batch = []

        if t < 3:
            grupos_ativos = [0, 1, 2]
            n_por_grupo = [7, 7, 6]

        else:
            grupos_ativos = [0, 1, 2, 3]
            n_por_grupo = [5, 5, 5, 18]

        for g, n in zip(grupos_ativos, n_por_grupo):
            df_g = gerar_grupo(
                parametros_stream[g], n
            )

            df_g["Grupo"] = g
            dados_batch.append(df_g)

        batch = pd.concat(
            dados_batch, ignore_index=True
        )

        batch["tempo"] = tempos_lotes[t]
        lotes.append(batch)

    return df_base, lotes





def gerar_base_2(seed):

    np.random.seed(seed)

    n_variaveis = 10
    obs_por_grupo = 100

    medias = [
        np.zeros(n_variaveis),
        np.ones(n_variaveis) * 5,
        np.ones(n_variaveis) * 10
    ]

    cov_identidade = np.identity(n_variaveis)

    cov_correlacionada = np.full(
        (n_variaveis, n_variaveis), 0.5
    )
    np.fill_diagonal(cov_correlacionada, 1)

    # -------------------------------
    # Base inicial
    # -------------------------------
    dados = []
    grupos = []

    for g in range(3):

        cov = (
            cov_correlacionada
            if g == 2
            else cov_identidade
        )

        cluster = np.random.multivariate_normal(
            mean=medias[g],
            cov=cov,
            size=obs_por_grupo
        )

        dados.append(cluster)
        grupos.extend([g] * obs_por_grupo)

    dados = np.vstack(dados)

    df_base = pd.DataFrame(
        dados,
        columns=[
            f"Var{i+1}" for i in range(n_variaveis)
        ]
    )

    df_base["Grupo"] = grupos
    df_base["tempo"] = 0

    # -------------------------------
    # Geração dos lotes
    # -------------------------------
    lotes = []
    tempos = list(range(1, 11))

    for t in range(10):

        if t < 3:
            # Três grupos antes do surgimento do novo
            dados_lote = np.vstack([
                np.random.normal(
                    0, 1, (15, n_variaveis)
                ),
                np.random.normal(
                    5, 1, (25, n_variaveis)
                ),
                np.random.normal(
                    10, 1, (10, n_variaveis)
                )
            ])

            grupos_lote = (
                [0] * 15 +
                [1] * 25 +
                [2] * 10
            )

        else:
            # Surgimento do grupo 3
            dados_lote = np.vstack([
                np.random.normal(
                    0, 1, (5, n_variaveis)
                ),
                np.random.normal(
                    5, 1, (10, n_variaveis)
                ),
                np.random.normal(
                    10, 1, (10, n_variaveis)
                ),
                np.random.normal(
                    15, 1, (25, n_variaveis)
                )
            ])

            grupos_lote = (
                [0] * 5 +
                [1] * 10 +
                [2] * 10 +
                [3] * 25
            )

        batch = pd.DataFrame(
            dados_lote,
            columns=[
                f"Var{i+1}" for i in range(n_variaveis)
            ]
        )

        batch["Grupo"] = grupos_lote
        batch["tempo"] = tempos[t]

        lotes.append(batch)

    return df_base, lotes


In [ ]:
N_REPLICACOES = 30

RESULTADOS_REPLICACOES = []
RESULTADOS_LOTES = []
for nome in SITUACOES_BRUTAS:
    cfg = ESCOLHAS_SIM[nome]
    gerador = gerar_base_2 if nome == 'Situação 1' else gerar_base_1
    for rep in range(N_REPLICACOES):
        seed = 1000 + rep
        base_rep, lotes_rep = gerador(seed)
        for metodo, resultado in [
            ('TID-KMeans', avaliar_tid_sim(base_rep, lotes_rep, **cfg['TID'], n_clusters=3)),
            ('DenStream', avaliar_denstream_sim(base_rep, lotes_rep, cfg['DenStream']))
        ]:
            RESULTADOS_REPLICACOES.append({
                'Situação':nome, 'Método':metodo, 'Replicação':rep+1, 'Seed':seed,
                'ARI_inicial':resultado['ARI_inicial'], 'ARI_stream':resultado['ARI_stream'],
                'ARI_global':resultado['ARI_global_com_base'], 'Clusters_finais':resultado['Numero_clusters_final']
            })
            for item in resultado['ARI_lotes']:
                RESULTADOS_LOTES.append({
                    'Situação':nome, 'Método':metodo, 'Replicação':rep+1, **item
                })

df_resultados = pd.DataFrame(RESULTADOS_REPLICACOES)
df_detalhados = pd.DataFrame(RESULTADOS_LOTES)
print(
    df_resultados
    .groupby(['Situação', 'Método'])[['ARI_inicial', 'ARI_stream', 'ARI_global']]
    .agg(['mean', 'std', 'min', 'max'])
    .round(4)
)
df_resultados.to_csv('replicacoes_padronizadas.csv',index=False)
df_detalhados.to_csv('replicacoes_lotes_padronizadas.csv',index=False)


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cores = {
    "TID-KMeans": "blue",
    "DenStream": "darkorange"
}

metodos = ["TID-KMeans", "DenStream"]
situacoes = ["Situação 1", "Situação 2"]

for situacao in situacoes:
    dados = df_resultados[
        df_resultados["Situação"] == situacao
    ]

    fig, ax = plt.subplots(figsize=(9, 4.5))

    for metodo in metodos:
        parte = dados[
            dados["Método"] == metodo
        ].sort_values("Replicação")

        ax.plot(
            parte["Replicação"],
            parte["ARI_global"],
            marker="o",
            markersize=4,
            linewidth=1.8,
            color=cores[metodo],
            label=metodo
        )

    ax.set_title(
        f"ARI global nas 30 replicações — {situacao}"
    )
    ax.set_xlabel("Replicação")
    ax.set_ylabel("ARI global")
    ax.set_xticks(range(1, 31))
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend()

    plt.tight_layout()

    fig.savefig(
        f"ari_global_{situacao.replace(' ', '_')}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


In [ ]:

resumo_lotes = (
    df_detalhados
    .groupby(["Situação", "Método", "Lote"])["ARI"]
    .agg(media="mean", desvio="std")
    .reset_index()
)

for situacao in situacoes:
    dados = resumo_lotes[
        resumo_lotes["Situação"] == situacao
    ]

    fig, ax = plt.subplots(figsize=(9, 4.5))

    for metodo in metodos:
        parte = dados[
            dados["Método"] == metodo
        ].sort_values("Lote")

        lotes = parte["Lote"].to_numpy()
        media = parte["media"].to_numpy()
        desvio = parte["desvio"].fillna(0).to_numpy()

        ax.plot(
            lotes,
            media,
            marker="o",
            markersize=5,
            linewidth=2,
            color=cores[metodo],
            label=metodo
        )

        ax.fill_between(
            lotes,
            np.clip(media - desvio, -1, 1),
            np.clip(media + desvio, -1, 1),
            color=cores[metodo],
            alpha=0.15
        )

    ax.set_title(
        f"ARI médio por lote nas 30 replicações — {situacao}"
    )
    ax.set_xlabel("Lote")
    ax.set_ylabel("ARI médio")
    ax.set_xticks(range(1, 11))
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend()

    plt.tight_layout()

    fig.savefig(
        f"ari_medio_lotes_{situacao.replace(' ', '_')}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
